# R23/R24 undersampling remedies - LLM tier gates

**Purpose**: adjudicate the LLM-clause hypotheses of the R23 undersampling-mechanism round and the R24 remedy slate, entirely as OFFLINE measurement against the frozen H119 extraction harness. No graph writes, no Neo4j, no Bedrock. The one live dependency is the local vLLM `gpt-oss-120b` at `localhost:8010` (temperature 0 unless a registration varies it).

Hypotheses (batch order, cheap gates first): **H246** enumerate-then-extract, **H248** GLiNER-primed pass, **H243** complement pass, **H258** mention-emission, **H251** parallel-sampling union, **H245/H257** logprob + top-k harvest, **H260** GLiNER adjudication, **H261** union-demo distillation. The registered designs, bars and gates in `docs/experiments/kgf-redesign-experiments.md` (sections R23, R24) are binding.

**Frozen harness (reused exactly from the R23/R24 CPU notebooks)**: gold carriers = the 101 unique `product` fields of `data/processed/probes-wide-v2-h195.json`; a name matches a carrier when `rapidfuzz.fuzz.token_set_ratio(name.lower(), product.lower()) >= 85`; union-of-5 (arm `A_production`, 5 runs, 10 docs) = 63 coverable carriers; single-run pooled mean = 76.8%, union-of-2 = 89.0%. Chunk texts from `data/interim/h119_chunks.pkl`.

**Extraction atom = entity pass + one gleaning round**, reproducing the frozen H119 pipeline (`extract_document` with `split_entity_relation=True, gleaning_rounds=1`); the relation pass is skipped because it adds no entities. This matters: an entity-only pass on these docs recovers only 1-9 carriers where the full pipeline recovers ~20 - the gleaning round is load-bearing. "1x single-pass cost" is therefore two LLM calls per chunk (entity + gleaning); every remedy's cost multiplier is measured against it.

**Serving caveats (measured up front)**: (1) the vLLM `n>1` sampling parameter returns HTTP 500 on this gpt-oss build (harmony parser: *"Unexpected token 200005 while expecting start token 200006"*), so parallel-sampling (H251) is emulated as K independent `n=1` temp-0.3 requests, priced under H251's registered shared-prefill cost model (entity prefill counted once per chunk). (2) Guided-JSON decoding fails with the same harmony error, so H257's FSM-name-slot clauses fall back to plain-call logprob harvesting. (3) gpt-oss spends the token budget in a reasoning channel before the JSON, so `max_tokens` must be generous (5000+) or the final channel truncates to empty; the call wrapper grows the budget adaptively on a truncated empty response.

In [1]:
# Imports - grouped by category
import os, re, json, glob, pickle, time                      # stdlib
from statistics import mean, median
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

from rapidfuzz import fuzz                                    # frozen token_set_ratio matcher
from openai import OpenAI                                     # raw vLLM client (logprobs, usage)
from rich.console import Console
from rich.table import Table
from rich import box

from knowledge_graph_foundry.models import normalize_name, Ontology
from knowledge_graph_foundry.extraction.prompts import entity_only_messages, gleaning_messages
from knowledge_graph_foundry.config import PROJ_ROOT

os.chdir(PROJ_ROOT)                                          # nbconvert runs from notebooks/
console = Console()
print("imports ok; cwd:", os.getcwd())

2026-07-08 15:31:14.480 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

Frozen parameters, endpoint, document selection, and the sanity anchors. The anchors MUST reproduce the frozen reference numbers before any adjudication is trusted.

**Document selection** (locked, honest rationale): the single-doc gates and the 3-doc arms use documents whose 5 arm-A runs all succeeded, so a measured coverage gap is genuine undersampling and not pipeline flakiness. `GATE_DOC` is the worst-coverage FULLY-SUCCESSFUL document (`3B_User-Manual`, single-run mean 44.7% of its 34-carrier union); the registered "worst-coverage doc" by raw single-run mean is `0-2019...pdf` but 4 of its 5 runs returned zero names, which would spuriously starve a complement/enumerate pass of residue. `TESTDOCS` spans low/mid/high single-run coverage. H261 uses its own frozen demo split from the R24b fanout report.

In [2]:
# --- Frozen configuration ---
THR = 85
ENDPOINT = "http://localhost:8010/v1"
MODEL = "gpt-oss-120b"
PURPOSE = "compare CPAP machines"
CKPT_GLOB = "results/h119/*.json"
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
PROBES = Path("data/processed/probes-wide-v2-h195.json")
GLINER_SPANS = Path("results/r23r24_llm/gliner_spans.json")
RLLM = Path("results/r23r24_llm"); RLLM.mkdir(parents=True, exist_ok=True)
REPORTS = Path("reports"); LOG_PATH = Path("logs/r23r24-llm-gates.log")

GATE_DOC = "3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf"
TESTDOCS = [GATE_DOC, "CPAP-Machines-Brochure.pdf", "CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf"]
DEMO_DOCS = ["CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf",
             "ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and_Performance)_Version_5.0_-_05-02-2022.pdf",
             "CPAP-Machines-Brochure.pdf"]   # frozen R24b split (fanout-gates-r24b)

client = OpenAI(base_url=ENDPOINT, api_key="local", timeout=180)

def log(msg):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with open(LOG_PATH, "a") as fh: fh.write(f"[{stamp}] {msg}\n")
    print(msg)

# --- Load checkpoints: arm -> run -> doc -> [names] ---
ARMS = {}
for f in glob.glob(CKPT_GLOB):
    d = json.load(open(f)); ARMS.setdefault(d["arm"], {}).setdefault(d["run"], {})[d["doc"]] = d["names"]
A = ARMS["A_production"]; RUNS = sorted(A); DOCS = sorted(A[1])

# --- Gold carriers + frozen matcher ---
PRODUCTS = list(dict.fromkeys(p["product"] for p in json.load(open(PROBES))["probes"]))
PL = [p.lower() for p in PRODUCTS]
def matched(names):
    el = [str(e).lower() for e in names]
    return frozenset(i for i, pr in enumerate(PL) if any(fuzz.token_set_ratio(e, pr) >= THR for e in el))

# --- Chunk texts ---
chunks = pickle.load(open(CHUNK_CACHE, "rb"))
DOCCHUNKS = {d: [c["text"] for c in sorted(chunks[d], key=lambda c: c["index"])] for d in DOCS}
DOCTEXT = {d: "".join(DOCCHUNKS[d]) for d in DOCS}

# --- per-(run,doc) matched carrier sets, doc-set union / baseline ---
M = {(r, d): matched(A[r].get(d, [])) for r in RUNS for d in DOCS}
def union5(D): return frozenset().union(*(M[(r, d)] for r in RUNS for d in D))
def baseline_mean(D):
    u = union5(D); n = len(u) or 1
    return mean(len(frozenset().union(*(M[(r, d)] for d in D)) & u) / n for r in RUNS)
def cov(names_by_doc, D):
    u = union5(D); n = len(u) or 1
    pooled = [nm for d in D for nm in names_by_doc.get(d, [])]
    return len(matched(pooled) & u) / n

UNION5 = union5(DOCS); U = len(UNION5)
single_pooled = mean(len(frozenset().union(*(M[(r, d)] for d in DOCS)) & UNION5) / U for r in RUNS)

# --- GLiNER cached spans (doc -> [(text, score)]) ---
GSPANS = json.load(open(GLINER_SPANS))
def gliner_names(doc, conf=0.30):
    return list(dict.fromkeys(t for (t, s) in GSPANS[doc] if s >= conf))

RESULTS = {}
_gall = [t2 for d in DOCS for t2 in gliner_names(d, 0.30)]
t = Table(title="Configuration and sanity anchors", box=box.SIMPLE)
t.add_column("key"); t.add_column("value"); t.add_column("expected")
t.add_row("arms x runs x docs", f"{len(ARMS)} x {len(RUNS)} x {len(DOCS)}", "3 x 5 x 10")
t.add_row("gold products (unique)", str(len(PRODUCTS)), "101")
t.add_row("union-of-5 coverable", str(U), "63")
t.add_row("single-run pooled mean", f"{single_pooled:.1%}", "76.8%")
t.add_row("GATE_DOC baseline (of its union)", f"{baseline_mean([GATE_DOC]):.1%}", "44.7%")
t.add_row("GLiNER recall @0.30", f"{len(matched(_gall) & UNION5)/U:.1%}", "93.7%")
console.print(t)
log(f"config ok: union5={U} single={single_pooled:.3f}")

               Configuration and sanity anchors               
                                                              
  key                                value        expected    
 ──────────────────────────────────────────────────────────── 
  arms x runs x docs                 3 x 5 x 10   3 x 5 x 10  
  gold products (unique)             101          101         
  union-of-5 coverable               63           63          
  single-run pooled mean             76.8%        76.8%       
  GATE_DOC baseline (of its union)   44.7%        44.7%       
  GLiNER recall @0.30                93.7%        93.7%

config ok: union5=63 single=0.768


## LLM helpers

The extraction atom `extract_chunk_names` is entity pass + one gleaning round with a given system prompt, mirroring the frozen pipeline. `run_pass` maps it over a document set (chunks in parallel) and records per-document token cost. `call` is a raw-call wrapper that grows `max_tokens` adaptively when gpt-oss over-reasons and truncates to empty content, and retries the harmony 500s. A resumable cache under `results/r23r24_llm/` preserves every expensive output across a crash.

In [3]:
JSON_SUFFIX = ("\n\nReturn ONLY a JSON object of the form "
               '{\"entities\": [{\"name\": \"...\"}], \"relationships\": []}. '
               "Use specific entity names; no commentary outside the JSON.")
_BASE_SYS = entity_only_messages("", PURPOSE, Ontology())[0]["content"] + JSON_SUFFIX
MAXTOK = 5000

def parse_names(content):
    if not content: return []
    s = content
    if "final<|message|>" in s: s = s.split("final<|message|>")[-1]
    s = re.sub(r"```(json)?", "", s)
    m = re.search(r"\{.*\}", s, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            ents = obj.get("entities", []) if isinstance(obj, dict) else obj
            out = [e["name"] if isinstance(e, dict) else e for e in ents]
            return [str(x) for x in out if x]
        except Exception:
            pass
    return re.findall(r'"name"\s*:\s*"([^"]+)"', s)

def _raw(system, user, temperature, max_tokens, logprobs, top_logprobs):
    kw = dict(model=MODEL, messages=[{"role": "system", "content": system},
                                     {"role": "user", "content": user}],
              temperature=temperature, max_tokens=max_tokens)
    if logprobs: kw.update(logprobs=True, top_logprobs=top_logprobs or 4)
    return client.chat.completions.create(**kw)

def call(system, user, temperature=0.0, max_tokens=MAXTOK, logprobs=False, top_logprobs=None):
    """Return (names, prompt_tokens, completion_tokens, response). Grows budget on truncated-empty."""
    mt = max_tokens; r = None
    for attempt in range(5):
        try:
            r = _raw(system, user, temperature, mt, logprobs, top_logprobs)
        except Exception:
            time.sleep(1.3 * (attempt + 1)); continue
        ch = r.choices[0]; names = parse_names(ch.message.content)
        if not names and ch.finish_reason == "length" and mt < 9000:
            mt = int(mt * 1.7); continue
        return names, r.usage.prompt_tokens, r.usage.completion_tokens, r
    if r is None: return [], 0, 0, None
    ch = r.choices[0]
    return parse_names(ch.message.content), r.usage.prompt_tokens, r.usage.completion_tokens, r

def map_chunks(fn, items, workers=4):
    out = [None] * len(items)
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(fn, it): i for i, it in enumerate(items)}
        for f in as_completed(futs): out[futs[f]] = f.result()
    return out

def cached(name, compute):
    p = RLLM / f"{name}.json"
    if p.exists():
        log(f"{name}: cached"); return json.load(open(p))
    log(f"{name}: computing"); t0 = time.time()
    v = compute(); v["_wall_s"] = round(time.time() - t0, 1)
    json.dump(v, open(p, "w")); return v

def extract_chunk_names(system, tx, temperature=0.0, glean=True):
    """Entity pass + one gleaning round -> (names, prompt_tokens, completion_tokens)."""
    e, pi, po, _ = call(system, tx, temperature=temperature)
    if not glean:
        return list(dict.fromkeys(e)), pi, po
    gsys = gleaning_messages("", e, PURPOSE, Ontology())[0]["content"] + JSON_SUFFIX
    g, gi, go, _ = call(gsys, tx, temperature=temperature)
    return list(dict.fromkeys(e + g)), pi + gi, po + go

def run_pass(system, D, temperature=0.0, glean=True):
    nbd, tok = {}, {}
    for d in D:
        res = map_chunks(lambda tx: extract_chunk_names(system, tx, temperature, glean), DOCCHUNKS[d])
        nbd[d] = [n for (nm, _, _) in res for n in nm]
        tok[d] = [sum(x[1] for x in res), sum(x[2] for x in res)]
    return {"names_by_doc": nbd, "tok": tok}

def pass_cost(P, D):
    return sum(P["tok"][d][0] + P["tok"][d][1] for d in D)

BASE_TEST = cached("basepass", lambda: run_pass(_BASE_SYS, TESTDOCS))
BASE_TEST_COST = pass_cost(BASE_TEST, TESTDOCS)
BASE_GATE_COST = BASE_TEST["tok"][GATE_DOC][0] + BASE_TEST["tok"][GATE_DOC][1]
print("base single-pass over TESTDOCS: cov=%.3f  tokens=%d  (checkpoint baseline %.3f)"
      % (cov(BASE_TEST["names_by_doc"], TESTDOCS), BASE_TEST_COST, baseline_mean(TESTDOCS)))

basepass: cached
base single-pass over TESTDOCS: cov=0.857  tokens=97371  (checkpoint baseline 0.834)


## R23-H246 - enumerate-then-extract

Split recall (stage 1: exhaustively LIST every entity NAME, terse) from generation (stage 2: extract details for the named candidates). Both stages are one call per chunk, so the operator costs ~1x the entity+gleaning baseline. **Gate** (1 doc, enumerate stage only, 3 runs temp 0.3): the enumeration union must exceed the doc's single-pass checkpoint coverage by >= 15 pts. **Full clause** (3 docs, one enumerate-then-extract pass): reach >= 80% of union-of-5 coverage at <= 1.5x single-pass token cost.

In [4]:
ENUM_SYS = ("You exhaustively LIST entity NAMES for a knowledge graph. Purpose: compare CPAP machines. "
            "List every distinct entity name present - products, device models, manufacturers, components, "
            "accessories, model codes, features. Be exhaustive; names only, do not describe types. " + JSON_SUFFIX)
def enum_doc(d, temperature, runs=1):
    per_run, pin, pout = [], 0, 0
    for _ in range(runs):
        res = map_chunks(lambda tx: call(ENUM_SYS, tx, temperature=temperature), DOCCHUNKS[d])
        per_run.append([n for (nm, _, _, _) in res for n in nm])
        pin += sum(x[1] for x in res); pout += sum(x[2] for x in res)
    return per_run, pin, pout

def h246_gate():
    per_run, pin, pout = enum_doc(GATE_DOC, 0.3, runs=3)
    union_names = [n for run in per_run for n in run]
    return {"enum_union_cov": cov({GATE_DOC: union_names}, [GATE_DOC]),
            "baseline": baseline_mean([GATE_DOC]), "pin": pin, "pout": pout}
g = cached("h246_gate", h246_gate)
g_pass = (g["enum_union_cov"] - g["baseline"]) >= 0.15
h246 = {"gate": {"enum_union_cov": g["enum_union_cov"], "baseline": g["baseline"],
                 "delta": g["enum_union_cov"]-g["baseline"], "bar": 0.15, "pass": bool(g_pass)}}
log(f"H246 gate: enum_union={g['enum_union_cov']:.3f} baseline={g['baseline']:.3f} pass={g_pass}")
if g_pass:
    def h246_full():
        nbd, pin, pout = {}, 0, 0
        for d in TESTDOCS:
            s1 = map_chunks(lambda tx: call(ENUM_SYS, tx, temperature=0.0), DOCCHUNKS[d])
            cand = [n for (nm, _, _, _) in s1 for n in nm]
            pin += sum(x[1] for x in s1); pout += sum(x[2] for x in s1)
            s2sys = _BASE_SYS + ("\n\nCandidate names detected in this text (extract those that are real entities, "
                                 "add any missed):\n" + "\n".join(f"- {c}" for c in cand[:150]))
            s2r = map_chunks(lambda tx: call(s2sys, tx, temperature=0.0), DOCCHUNKS[d])
            pin += sum(x[1] for x in s2r); pout += sum(x[2] for x in s2r)
            nbd[d] = list(dict.fromkeys(cand + [n for (nm, _, _, _) in s2r for n in nm]))
        return {"names_by_doc": nbd, "pin": pin, "pout": pout}
    fr = cached("h246_full", h246_full)
    ccov = cov(fr["names_by_doc"], TESTDOCS); mult = (fr["pin"] + fr["pout"]) / BASE_TEST_COST
    h246["full"] = {"cov": ccov, "cov_bar": 0.80, "cost_mult": mult, "cost_bar": 1.5,
                    "pass": bool(ccov >= 0.80 and mult <= 1.5), "tokens": fr["pin"]+fr["pout"]}
    log(f"H246 full: cov={ccov:.3f} cost={mult:.2f}x")
RESULTS["H246"] = h246
print(json.dumps(h246, indent=1))

h246_gate: cached
H246 gate: enum_union=1.000 baseline=0.447 pass=True
h246_full: cached
H246 full: cov=0.971 cost=1.05x
{
 "gate": {
  "enum_union_cov": 1.0,
  "baseline": 0.4470588235294118,
  "delta": 0.5529411764705883,
  "bar": 0.15,
  "pass": true
 },
 "full": {
  "cov": 0.9714285714285714,
  "cov_bar": 0.8,
  "cost_mult": 1.0512781012827228,
  "cost_bar": 1.5,
  "pass": true,
  "tokens": 102364
 }
}


## R23-H248 - GLiNER-primed single pass

The GLiNER lexicon (conf 0.30) primes the recall stage: each chunk's extraction (entity + gleaning) is cued with the document's detected surface forms. **LLM clause**: a primed single pass on 3 docs lifts carrier coverage >= 20 pts over the unprimed single-pass baseline.

In [5]:
def h248():
    nbd, tok = {}, {}
    for d in TESTDOCS:
        cues = gliner_names(d, conf=0.30)
        sysd = _BASE_SYS + ("\n\nEnsure you cover these detected candidate surface forms if they denote real "
                            "entities in the text:\n" + "\n".join(f"- {c}" for c in cues[:150]))
        res = map_chunks(lambda tx: extract_chunk_names(sysd, tx, 0.0, True), DOCCHUNKS[d])
        nbd[d] = [n for (nm, _, _) in res for n in nm]
        tok[d] = [sum(x[1] for x in res), sum(x[2] for x in res)]
    return {"names_by_doc": nbd, "tok": tok}
r = cached("h248", h248)
primed = cov(r["names_by_doc"], TESTDOCS)
unprimed = cov(BASE_TEST["names_by_doc"], TESTDOCS)
unprimed_ckpt = baseline_mean(TESTDOCS)
RESULTS["H248"] = {"primed_cov": primed, "unprimed_cov_control": unprimed, "unprimed_cov_ckpt": unprimed_ckpt,
                   "lift_vs_control": primed-unprimed, "lift_vs_ckpt": primed-unprimed_ckpt, "bar": 0.20,
                   "pass": bool((primed-unprimed_ckpt) >= 0.20 or (primed-unprimed) >= 0.20),
                   "cost_mult": pass_cost(r, TESTDOCS)/BASE_TEST_COST}
log(f"H248: primed={primed:.3f} unprimed_ctrl={unprimed:.3f} ckpt={unprimed_ckpt:.3f}")
print(json.dumps(RESULTS["H248"], indent=1))

h248: cached
H248: primed=0.943 unprimed_ctrl=0.857 ckpt=0.834
{
 "primed_cov": 0.9428571428571428,
 "unprimed_cov_control": 0.8571428571428571,
 "unprimed_cov_ckpt": 0.8342857142857143,
 "lift_vs_control": 0.08571428571428574,
 "lift_vs_ckpt": 0.10857142857142854,
 "bar": 0.2,
 "pass": false,
 "cost_mult": 1.2579515461482371
}


## R23-H243 - complement pass

A directed complement pass ("list entities present in the text but absent from this list") should recover the residual better than an iid second pass. **Gate** (worst doc, complement against run-1 checkpoint): recover >= 1 new union-verified carrier. **Full clause** (3 docs): complement recovers >= 1.3x the NEW gold carriers of an iid second pass (per-token also reported, since the complement is one call vs the iid pass's entity+gleaning).

In [6]:
COMP_SYS = ("You review a knowledge-graph extraction and find what was MISSED. Purpose: compare CPAP machines. "
            "List entities present in the text but NOT already in the provided list - products, devices, "
            "manufacturers, components, accessories, model codes, features. " + JSON_SUFFIX)
def complement_doc(d, existing, temperature=0.0):
    sysd = COMP_SYS + "\n\nAlready extracted (do NOT repeat):\n" + "\n".join(
        f"- {n}" for n in list(dict.fromkeys(existing))[:200])
    res = map_chunks(lambda tx: call(sysd, tx, temperature=temperature), DOCCHUNKS[d])
    return [n for (nm, _, _, _) in res for n in nm], sum(x[1]+x[2] for x in res)

def h243_gate():
    pass1 = A[1].get(GATE_DOC, [])
    comp, cost = complement_doc(GATE_DOC, pass1)
    u = union5([GATE_DOC]); new = (matched(comp) & u) - (matched(pass1) & u)
    return {"pass1_n": len(pass1), "comp_new_carriers": len(new), "cost": cost}
g = cached("h243_gate", h243_gate)
h243 = {"gate": {"comp_new_carriers": g["comp_new_carriers"], "bar": 1, "pass": bool(g["comp_new_carriers"] >= 1)}}
log(f"H243 gate: comp_new={g['comp_new_carriers']}")
if h243["gate"]["pass"]:
    def h243_full():
        out = {}
        for d in TESTDOCS:
            p1 = run_pass(_BASE_SYS, [d], 0.0); pass1 = p1["names_by_doc"][d]
            p2 = run_pass(_BASE_SYS, [d], 0.3); iid = p2["names_by_doc"][d]
            comp, ccost = complement_doc(d, pass1)
            out[d] = {"pass1": pass1, "iid": iid, "comp": comp,
                      "cost_iid": pass_cost(p2, [d]), "cost_comp": ccost}
        return out
    fr = cached("h243_full", h243_full)
    new_iid = new_comp = 0
    for d in TESTDOCS:
        u = union5([d]); m1 = matched(fr[d]["pass1"]) & u
        new_iid += len((matched(fr[d]["iid"]) & u) - m1)
        new_comp += len((matched(fr[d]["comp"]) & u) - m1)
    cost_iid = sum(fr[d]["cost_iid"] for d in TESTDOCS); cost_comp = sum(fr[d]["cost_comp"] for d in TESTDOCS)
    ratio = new_comp / new_iid if new_iid else None
    per_tok = (new_comp/cost_comp) / (new_iid/cost_iid) if new_iid and cost_comp and cost_iid else None
    h243["full"] = {"new_iid": new_iid, "new_comp": new_comp, "ratio": ratio, "ratio_per_token": per_tok,
                    "bar": 1.3, "cost_iid": cost_iid, "cost_comp": cost_comp,
                    "pass": bool(new_comp >= 1.3 * new_iid)}
    log(f"H243 full: new_comp={new_comp} new_iid={new_iid} ratio={ratio}")
RESULTS["H243"] = h243
print(json.dumps(h243, indent=1))

h243_gate: cached
H243 gate: comp_new=0
{
 "gate": {
  "comp_new_carriers": 0,
  "bar": 1,
  "pass": false
 }
}


## R24-H258 - mention-emission

Move dedup out of the model: emit every surface MENTION and let the resolver aggregate. The **free gate** already CONFIRMED (81.4% cross-document known-but-dropped). **LLM clause**: mention-emission single pass reaches >= 80% carrier coverage at ~2-3x mention inflation, with resolver precision loss <= 5 pts. The resolver-precision clause needs the live v2 resolver + a graph write, which this offline batch forbids - it is recorded UNTESTABLE-offline; coverage and mention inflation are measured. Mention emission is one call per chunk (no gleaning - the prompt already forbids deduplication).

In [7]:
MENTION_SYS = ("You extract every surface MENTION of every entity for a knowledge graph. Purpose: compare CPAP machines. "
               "Emit each entity mention AS IT APPEARS in the text, including repeated and variant mentions of the same "
               "entity - do NOT deduplicate or canonicalize; a downstream resolver will merge duplicates. " + JSON_SUFFIX)
def h258():
    nbd, tok = {}, {}
    for d in TESTDOCS:
        res = map_chunks(lambda tx: call(MENTION_SYS, tx, temperature=0.0, max_tokens=6500), DOCCHUNKS[d])
        nbd[d] = [n for (nm, _, _, _) in res for n in nm]
        tok[d] = [sum(x[1] for x in res), sum(x[2] for x in res)]
    return {"names_by_doc": nbd, "tok": tok}
r = cached("h258", h258)
mcov = cov(r["names_by_doc"], TESTDOCS)
allm = [n for d in TESTDOCS for n in r["names_by_doc"][d]]
uniq = len(set(normalize_name(n) for n in allm))
inflation = len(allm) / uniq if uniq else 0
base_uniq = len(set(normalize_name(n) for d in TESTDOCS for n in BASE_TEST["names_by_doc"][d]))
RESULTS["H258"] = {"cov": mcov, "cov_bar": 0.80, "cov_pass": bool(mcov >= 0.80),
                   "mentions": len(allm), "unique": uniq, "inflation": inflation,
                   "base_unique": base_uniq, "inflation_vs_base": len(allm)/max(base_uniq, 1),
                   "resolver_precision_clause": "UNTESTABLE-offline (needs v2 resolver + graph write)",
                   "cost_mult": pass_cost(r, TESTDOCS)/BASE_TEST_COST}
log(f"H258: cov={mcov:.3f} inflation={inflation:.2f}")
print(json.dumps(RESULTS["H258"], indent=1))

h258: cached
H258: cov=1.000 inflation=1.53
{
 "cov": 1.0,
 "cov_bar": 0.8,
 "cov_pass": true,
 "mentions": 1121,
 "unique": 732,
 "inflation": 1.5314207650273224,
 "base_unique": 275,
 "inflation_vs_base": 4.076363636363636,
 "resolver_precision_clause": "UNTESTABLE-offline (needs v2 resolver + graph write)",
 "cost_mult": 0.7590453009622988
}


## R24-H251 - parallel-sampling union

K samples for ~1.3 passes on a shared prefill. **Serving limitation**: the vLLM `n>1` parameter 500s on this gpt-oss build (harmony), so K=5 is emulated as 5 independent temp-0.3 entity+gleaning passes; cost is priced under H251's registered shared-prefill model (the entity prefill counted ONCE per chunk + all decodes; the gleaning call does not share prefill and is counted in full). **Gate** (1 doc): n-sample union within 10 pts of the independent union-of-5 AND cost <= 2x. **Full**: (a) n=5 temp 0.3 on 3 docs >= 95% of the independent union-of-5 coverage at <= 1.6x; (b) temp 0.3 exceeds temp~0 by >= 10 pts.

In [8]:
def nsample_doc(d, temperature, K=5):
    """K entity+gleaning samples per chunk; shared-prefill cost = entity-prefill once + all decodes + gleaning."""
    per_sample = [[] for _ in range(K)]; cost_shared = 0
    for tx in DOCCHUNKS[d]:
        res = map_chunks(lambda _k: extract_chunk_names(_BASE_SYS, tx, temperature, True), list(range(K)))
        in_once = res[0][1]; decodes = sum(x[2] for x in res)   # entity prefill shared; decodes+gleaning counted
        cost_shared += in_once + decodes
        for k in range(K): per_sample[k].extend(res[k][0])
    return per_sample, cost_shared

def indep_union_cov(d):
    u = union5([d])
    return (len(frozenset().union(*(M[(r, d)] for r in RUNS)) & u) / len(u)) if u else 0.0

def h251_gate():
    ps, cost = nsample_doc(GATE_DOC, 0.3, K=5)
    unames = [n for s in ps for n in s]
    return {"nsample_cov": cov({GATE_DOC: unames}, [GATE_DOC]),
            "indep_union_cov": indep_union_cov(GATE_DOC), "cost_shared": cost}
g = cached("h251_gate", h251_gate)
gate_cost_mult = g["cost_shared"] / BASE_GATE_COST
gate_pass = abs(g["nsample_cov"] - g["indep_union_cov"]) <= 0.10 and gate_cost_mult <= 2.0
h251 = {"gate": {"nsample_cov": g["nsample_cov"], "indep_union_cov": g["indep_union_cov"],
                 "cov_gap": g["nsample_cov"]-g["indep_union_cov"], "cost_mult": gate_cost_mult,
                 "pass": bool(gate_pass)}}
log(f"H251 gate: nsamp={g['nsample_cov']:.3f} indep={g['indep_union_cov']:.3f} cost={gate_cost_mult:.2f}x")
if gate_pass:
    def h251_full():
        hot = {"cost": 0, "nbd": {}}; cold = {"cost": 0, "nbd": {}}
        for d in TESTDOCS:
            ps, c = nsample_doc(d, 0.3, K=5); hot["cost"] += c; hot["nbd"][d] = [n for s in ps for n in s]
            ps2, c2 = nsample_doc(d, 0.0, K=5); cold["cost"] += c2; cold["nbd"][d] = [n for s in ps2 for n in s]
        return {"hot": hot, "cold": cold}
    fr = cached("h251_full", h251_full)
    hot_cov = cov(fr["hot"]["nbd"], TESTDOCS); cold_cov = cov(fr["cold"]["nbd"], TESTDOCS)
    indep_set = cov({d: [n for r in RUNS for n in A[r].get(d, [])] for d in TESTDOCS}, TESTDOCS)
    mult = fr["hot"]["cost"] / BASE_TEST_COST
    h251["full"] = {"hot_cov": hot_cov, "cold_cov": cold_cov, "indep_union5_cov": indep_set,
                    "pct_of_indep": hot_cov/indep_set if indep_set else 0, "temp_delta": hot_cov-cold_cov,
                    "cost_mult": mult, "clause_a_pass": bool(hot_cov >= 0.95*indep_set and mult <= 1.6),
                    "clause_b_pass": bool((hot_cov-cold_cov) >= 0.10)}
    log(f"H251 full: hot={hot_cov:.3f} cold={cold_cov:.3f} indep={indep_set:.3f} cost={mult:.2f}x")
RESULTS["H251"] = h251
print(json.dumps(h251, indent=1))

h251_gate: cached
H251 gate: nsamp=0.912 indep=1.000 cost=3.05x
{
 "gate": {
  "nsample_cov": 0.9117647058823529,
  "indep_union_cov": 1.0,
  "cov_gap": -0.08823529411764708,
  "cost_mult": 3.0495843819110804,
  "pass": false
 }
}


## R23-H245 + R24-H257 - logprob signature and top-k harvest

1 document x 3 entity passes with per-token logprobs (top-4 alternatives) captured (no gleaning - the logprob signature is measured on the entity pass). **H245**: churn-class entity names (present in < 3 of the 5 frozen checkpoints) sit >= 0.5 nats below the stable core's mean name-span logprob. **H257(a)**: top-k alternatives at name-value positions contain the missing carriers (best-effort - the registered FSM/guided-decoding path 500s on this build). **H257(b)** value-slot min-p sampling requires a logit processor over guided decoding: UNTESTABLE on this build.

In [9]:
def name_token_logprobs(resp):
    ch = resp.choices[0]; lp = ch.logprobs.content if ch.logprobs else None
    content = ch.message.content or ""; names = parse_names(content)
    if not lp: return [(n, None, []) for n in names]
    toks = [t.token for t in lp]; recon = ""; offs = []
    for tk in toks: offs.append(len(recon)); recon += tk
    out = []
    for n in names:
        pos = recon.rfind(n)
        if pos < 0: out.append((n, None, [])); continue
        end = pos + len(n); idxs = [i for i, o in enumerate(offs) if pos <= o < end]
        if not idxs: out.append((n, None, [])); continue
        mlp = mean(lp[i].logprob for i in idxs)
        alts = [(a.token, a.logprob) for a in (lp[idxs[0]].top_logprobs or [])]
        out.append((n, mlp, alts))
    return out

def h245_257():
    runs = []
    for _ in range(3):
        rundata = []
        for tx in DOCCHUNKS[GATE_DOC]:
            _, i, o, resp = call(_BASE_SYS, tx, temperature=0.0, logprobs=True, top_logprobs=4, max_tokens=6000)
            if resp is None: continue
            for (nm, mlp, alts) in name_token_logprobs(resp):
                rundata.append({"name": nm, "mlp": mlp, "alts": [[a, round(b, 3)] for a, b in alts]})
        runs.append(rundata)
    return {"runs": runs}
r = cached("h245_257", h245_257)

ckpt_counts = Counter()
for run in RUNS:
    for nm in set(normalize_name(x) for x in A[run].get(GATE_DOC, [])): ckpt_counts[nm] += 1
stable_lp, churn_lp = [], []
for run in r["runs"]:
    for e in run:
        if e["mlp"] is None: continue
        c = ckpt_counts.get(normalize_name(e["name"]), 0)
        (stable_lp if c >= 3 else churn_lp).append(e["mlp"])
sep = (mean(stable_lp) - mean(churn_lp)) if stable_lp and churn_lp else None
RESULTS["H245"] = {"stable_n": len(stable_lp), "churn_n": len(churn_lp),
                   "stable_mean_lp": mean(stable_lp) if stable_lp else None,
                   "churn_mean_lp": mean(churn_lp) if churn_lp else None,
                   "separation_nats": sep, "bar": 0.5, "pass": bool(sep is not None and sep >= 0.5)}
log(f"H245: sep={sep}")
pass_names = [e["name"] for run in r["runs"] for e in run]
u = union5([GATE_DOC]); missing = u - (matched(pass_names) & u)
alt_tokens = [a.strip() for run in r["runs"] for e in run for (a, b) in e["alts"]]
alt_hits = matched(alt_tokens) & missing
RESULTS["H257"] = {"clause_a": {"missing_carriers": len(missing), "alt_token_carrier_hits": len(alt_hits),
                                "note": "best-effort plain-call harvest; registered FSM name-slot path 500s (guided_json harmony bug)"},
                   "clause_b": "UNTESTABLE (min-p value-slot logit processor requires guided decoding, which 500s on this build)"}
print(json.dumps({"H245": RESULTS["H245"], "H257": RESULTS["H257"]}, indent=1))

h245_257: cached
H245: sep=0.006627077866027911


{
 "H245": {
  "stable_n": 183,
  "churn_n": 134,
  "stable_mean_lp": -0.001974538417106894,
  "churn_mean_lp": -0.008601616283134805,
  "separation_nats": 0.006627077866027911,
  "bar": 0.5,
  "pass": false
 },
 "H257": {
  "clause_a": {
   "missing_carriers": 29,
   "alt_token_carrier_hits": 10,
   "note": "best-effort plain-call harvest; registered FSM name-slot path 500s (guided_json harmony bug)"
  },
  "clause_b": "UNTESTABLE (min-p value-slot logit processor requires guided decoding, which 500s on this build)"
 }
}


## R24-H260 - GLiNER adjudication clause

Convert the LLM from open-vocabulary recall to closed-set adjudication: feed the GLiNER candidate spans (conf 0.30) and let one LLM pass keep the genuine entities and add any it sees missing (one call per chunk, no gleaning). **Clause**: GLiNER candidates + one adjudication pass >= 89% of union-of-5 (union-of-2 parity at ~1x LLM cost). Run over all 10 documents for a corpus-level number comparable to the 89% union-of-2 and 95.2% GLiNER recall anchors.

In [10]:
ADJ_SYS = ("You adjudicate candidate entity spans for a knowledge graph. Purpose: compare CPAP machines. "
           "You are given candidate surface forms detected in the text. Return the ones that denote GENUINE entities "
           "serving the purpose (products, devices, manufacturers, components, accessories, model codes, features), "
           "and ADD any real entity you see that is missing from the candidate list. " + JSON_SUFFIX)
def h260():
    nbd, pin, pout = {}, 0, 0
    for d in DOCS:
        cues = gliner_names(d, conf=0.30)
        sysd = ADJ_SYS + "\n\nCandidate spans:\n" + "\n".join(f"- {c}" for c in cues[:200])
        res = map_chunks(lambda tx: call(sysd, tx, temperature=0.0, max_tokens=6000), DOCCHUNKS[d])
        nbd[d] = [n for (nm, _, _, _) in res for n in nm]
        pin += sum(x[1] for x in res); pout += sum(x[2] for x in res)
    return {"names_by_doc": nbd, "pin": pin, "pout": pout}
r = cached("h260", h260)
acov = cov(r["names_by_doc"], DOCS)
RESULTS["H260"] = {"adjudicated_cov": acov, "bar": 0.89, "pass": bool(acov >= 0.89),
                   "union2_ref": 0.89, "gliner_recall_ref": 0.952, "tokens": r["pin"]+r["pout"]}
log(f"H260: adjudicated_cov={acov:.3f}")
print(json.dumps(RESULTS["H260"], indent=1))

h260: cached


H260: adjudicated_cov=0.873
{
 "adjudicated_cov": 0.873015873015873,
 "bar": 0.89,
 "pass": false,
 "union2_ref": 0.89,
 "gliner_recall_ref": 0.952,
 "tokens": 87209
}


## R24-H261 - union-demo distillation

Recalibrate the exhaustivity prior with 2-3 held-out union demonstrations in the prompt. Demos are the frozen low-leakage R24b split (CPAP-V3, ARTP, CPAP-Machines); targets are the other 7 documents (the cleanest demo/target pairs share zero gold carriers). Each target is a primed entity+gleaning pass. **Clause**: reach >= 78% carrier coverage (>= +20 pts over baseline) at 1 pass + ~2k prompt tokens, leakage controlled.

In [11]:
def demo_block():
    blocks = []
    for d in DEMO_DOCS:
        excerpt = DOCTEXT[d][:1400]
        exhaustive = list(dict.fromkeys(n for r in RUNS for n in A[r].get(d, [])))
        blocks.append("Example of EXHAUSTIVE extraction from a similar document:\nTEXT:\n" + excerpt
                      + "\nEXHAUSTIVE ENTITIES:\n" + ", ".join(exhaustive[:70]))
    return "\n\n".join(blocks)
DEMO = demo_block()
TARGETS = [d for d in DOCS if d not in DEMO_DOCS]
def h261():
    sysd = _BASE_SYS + "\n\nStudy these demonstrations of exhaustive extraction, then extract with the same exhaustivity:\n\n" + DEMO
    P = run_pass(sysd, TARGETS, 0.0, True)
    P["demo_tokens"] = len(DEMO)//4
    return P
r = cached("h261", h261)
dcov = cov(r["names_by_doc"], TARGETS); dbase = baseline_mean(TARGETS)
RESULTS["H261"] = {"distilled_cov": dcov, "baseline": dbase, "lift": dcov-dbase,
                   "abs_bar": 0.78, "lift_bar": 0.20, "demo_tokens_est": r["demo_tokens"],
                   "pass": bool(dcov >= 0.78 and (dcov-dbase) >= 0.20)}
log(f"H261: distilled={dcov:.3f} baseline={dbase:.3f} lift={dcov-dbase:.3f}")
print(json.dumps(RESULTS["H261"], indent=1))

h261: cached
H261: distilled=0.517 baseline=0.670 lift=-0.153
{
 "distilled_cov": 0.5166666666666667,
 "baseline": 0.67,
 "lift": -0.15333333333333332,
 "abs_bar": 0.78,
 "lift_bar": 0.2,
 "demo_tokens_est": 2322,
 "pass": false
}


## Cost frontier and report

Coverage-per-token frontier for every LLM remedy against the two reference operators (union-of-2 = 89.0% of union-5 at 2.0x; single pass = 76.8% at 1.0x), and the serialized report. "Beats union-of-2 on coverage-per-token" = coverage/cost_mult >= 0.89/2.0 = 0.445.

In [12]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {"round": "R23/R24", "tier": "LLM gates (offline, local vLLM gpt-oss-120b)",
          "generated_utc": stamp, "endpoint": ENDPOINT, "model": MODEL,
          "gold_carrier_definition": "token_set_ratio(name.lower(), product.lower()) >= 85; union-of-5 = 63",
          "extraction_atom": "entity pass + 1 gleaning round (reproduces frozen H119 pipeline)",
          "docsets": {"GATE_DOC": GATE_DOC, "TESTDOCS": TESTDOCS, "DEMO_DOCS": DEMO_DOCS},
          "anchors": {"union5": U, "single_pooled": single_pooled, "union2_ref": 0.89},
          "serving_caveats": ["vLLM n>1 -> HTTP 500 (harmony 200005/200006); n-sampling emulated as K separate calls",
                              "guided_json -> HTTP 500 (same harmony bug); H257 FSM clauses fall back / UNTESTABLE",
                              "gpt-oss over-reasons; max_tokens grown adaptively to avoid empty-truncation"],
          "base_testdocs_tokens": BASE_TEST_COST, "base_gatedoc_tokens": BASE_GATE_COST,
          "results": RESULTS}
outp = REPORTS / f"undersampling-llm-gates-r23r24-{stamp}.json"
json.dump(report, open(outp, "w"), indent=1)
log(f"report written: {outp}")

def verdict(p): return "[green]PASS[/green]" if p else "[red]FAIL[/red]"
t = Table(title="R23/R24 LLM-tier verdicts", box=box.SIMPLE_HEAVY)
for col in ["hypothesis", "key metric", "bar", "verdict"]: t.add_column(col)
rows = []
if "full" in RESULTS.get("H246", {}):
    f = RESULTS["H246"]["full"]; rows.append(("H246 enumerate", f"cov {f['cov']:.1%} @ {f['cost_mult']:.2f}x", ">=80% @<=1.5x", verdict(f["pass"])))
else:
    gg = RESULTS["H246"]["gate"]; rows.append(("H246 gate", f"enum union +{gg['delta']:.1%}", ">=+15pts", verdict(gg["pass"])))
h = RESULTS["H248"]; rows.append(("H248 primed", f"+{h['lift_vs_ckpt']:.1%} vs ckpt", ">=+20pts", verdict(h["pass"])))
h = RESULTS["H243"]
if "full" in h: rows.append(("H243 complement", f"ratio {h['full']['ratio']}", ">=1.3x", verdict(h["full"]["pass"])))
else: rows.append(("H243 gate", f"{h['gate']['comp_new_carriers']} new", ">=1", verdict(h["gate"]["pass"])))
h = RESULTS["H258"]; rows.append(("H258 mention", f"cov {h['cov']:.1%} infl {h['inflation']:.1f}x", ">=80% cov", verdict(h["cov_pass"])))
h = RESULTS["H251"]
if "full" in h:
    f = h["full"]; rows.append(("H251 n-sample(a)", f"{f['pct_of_indep']:.0%} indep @{f['cost_mult']:.2f}x", ">=95% @<=1.6x", verdict(f["clause_a_pass"])))
    rows.append(("H251 temp(b)", f"delta {f['temp_delta']:+.1%}", ">=+10pts", verdict(f["clause_b_pass"])))
else:
    gg = h["gate"]; rows.append(("H251 gate", f"gap {gg['cov_gap']:+.1%} @{gg['cost_mult']:.2f}x", "<=10pts @<=2x", verdict(gg["pass"])))
h = RESULTS["H245"]; rows.append(("H245 logprob", f"sep {h['separation_nats']}", ">=0.5 nats", verdict(h["pass"])))
h = RESULTS["H260"]; rows.append(("H260 adjudicate", f"cov {h['adjudicated_cov']:.1%}", ">=89%", verdict(h["pass"])))
h = RESULTS["H261"]; rows.append(("H261 distill", f"cov {h['distilled_cov']:.1%} (+{h['lift']:.1%})", ">=78% & +20pts", verdict(h["pass"])))
for row in rows: t.add_row(*row)
console.print(t)
print("REPORT:", outp)

report written: reports/undersampling-llm-gates-r23r24-20260708T133115Z.json


                        R23/R24 LLM-tier verdicts                        
                                                                         
  hypothesis        key metric                 bar              verdict  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  H246 enumerate    cov 97.1% @ 1.05x          >=80% @<=1.5x    PASS     
  H248 primed       +10.9% vs ckpt             >=+20pts         FAIL     
  H243 gate         0 new                      >=1              FAIL     
  H258 mention      cov 100.0% infl 1.5x       >=80% cov        PASS     
  H251 gate         gap -8.8% @3.05x           <=10pts @<=2x    FAIL     
  H245 logprob      sep 0.006627077866027911   >=0.5 nats       FAIL     
  H260 adjudicate   cov 87.3%                  >=89%            FAIL     
  H261 distill      cov 51.7% (+-15.3%)        >=78% & +20pts   FAIL

REPORT: reports/undersampling-llm-gates-r23r24-20260708T133115Z.json
